# Consultas do case

## Clientes com crescimento acima de 50% entre dois meses consecutivos

In [0]:
%sql

with mensal as (
select
    id_cliente,
    mes,
    valor_total,
    lag(valor_total) over (partition by id_cliente order by mes) as valor_mes_anterior
from novarota.gold.gold_resumo_cliente_mes)


select
    id_cliente,
    add_months(mes, -1) as mes_anterior,
    mes,
    valor_mes_anterior,
    valor_total,
    round((valor_total - valor_mes_anterior) / valor_mes_anterior * 100, 2) as crescimento_pct
from mensal
where valor_mes_anterior > 0
  and valor_total > valor_mes_anterior * 1.5
order by crescimento_pct desc

## Quartis de valor movimentado

In [0]:
%sql

with total_cliente as (
select
    id_cliente,
    sum(valor_total) as valor_movimentado
from novarota.gold.gold_resumo_cliente_mes
group by id_cliente)


select
    id_cliente,
    valor_movimentado,
    ntile(4) over (order by valor_movimentado desc) as quartil
from total_cliente
order by quartil, valor_movimentado desc